# Ableton × Claude · Live MIDI Generation Pipeline

Generate a fresh 10-track MIDI pattern every 16 bars with the Claude API, stream it to Ableton in real time via mido, and accumulate the clips into the Session View via AbletonOSC.

```
 ┌────────────┐   16-bar MIDI    ┌───────────┐   mido       ┌─────────┐
 │ Claude API │─────────────────▶│ Sequencer │─────────────▶│ Ableton │ (audio out)
 └────────────┘  (10 styles)     └───────────┘  10 channels └─────────┘
                                       │                          ▲
                                       │ AbletonOSC                │
                                       └──────────────────────────┘
                                       create_clip / add notes
                                       (Session View accumulation)
```

## Dependencies
```bash
pip install anthropic mido python-rtmidi matplotlib numpy python-osc python-dotenv
```

## API Key
Create a `.env` file in the project folder:
```
ANTHROPIC_API_KEY=sk-ant-xxxxx
```
or set `export ANTHROPIC_API_KEY=...` in your shell.

## Ableton one-time setup
1. **Virtual MIDI port**: macOS `Audio MIDI Setup` → IAC Driver → check `Device is online` → enable Bus 1
2. **Ableton MIDI preferences**: `Preferences → Link/Tempo/MIDI` → turn the **Track** toggle ON for the `IAC Driver (Bus 1)` row
3. **10 MIDI tracks**: each track `MIDI From: IAC Driver (Bus 1) / Ch. 1-10`, **Monitor: In**
4. **(optional) AbletonOSC** (for track colors + automatic Session View clip saving):
   - Download https://github.com/ideoforms/AbletonOSC and unzip it
   - Place the folder (named exactly `AbletonOSC`) into `~/Music/Ableton/User Library/Remote Scripts/`
   - Restart Ableton → select AbletonOSC under `Preferences → Link/Tempo/MIDI → Control Surface`
5. Set `CONFIG['bpm']` in the notebook = Ableton tempo

## Run order
Cell 1 → 2 → 3 → 4 → 5 → (6 optional) → 7 → 8 → 9 → 10 or 11

## Cell 1 · Imports & Setup

In [1]:
# API key is loaded from a .env file (never hardcode keys in the notebook)
import os
import json
import time
import random
import threading
import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import mido
from anthropic import Anthropic

# Load ANTHROPIC_API_KEY from a .env file (if present)
try:
    from dotenv import load_dotenv
    load_dotenv()
    print("✓ .env loaded" if os.environ.get("ANTHROPIC_API_KEY") else "⚠ .env loaded but ANTHROPIC_API_KEY not found")
except ImportError:
    print("  python-dotenv not installed -- reading shell environment variables directly")


# OSC (optional)
try:
    from pythonosc import udp_client, dispatcher, osc_server
    HAS_OSC = True
except ImportError:
    HAS_OSC = False
    print("! python-osc not installed -- skipping Section 6 (OSC)")

# Timestamped folder -- where all outputs (MIDI JSON, piano roll HTML, raw responses) are saved
TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path(f"outputs/{TIMESTAMP}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR.resolve()}")

# Anthropic client
client = Anthropic()
print(f"API key: {'sk-ant-' + os.environ.get('ANTHROPIC_API_KEY', '')[7:14] + '...' if os.environ.get('ANTHROPIC_API_KEY') else 'MISSING'}")

# matplotlib defaults (kept for any ad-hoc plotting)
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.linewidth": 0.6,
    "axes.edgecolor": "#333",
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

# Global handles -- filled in by optional components
midi_out = None
lq = None
osc_client = None

✓ .env loaded
Output directory: /Users/faacia/longlive/outputs/20260601_172101
API key: sk-ant-api03-_...


## Cell 2 · Configuration

Variables: `bpm`, `genre`, `mood`, `chord_progression` (4 chords, 4 bars each), `bars`, `key`. The track layout also specifies channel, pitch range, and role.

In [2]:
CONFIG = {
    "bpm": 134,
    "genre": "driving, danceable house — peak-time deep/tech house with organic percussion (Make A Dance / Jimpster energy, dancefloor-focused)",
    "mood": "energetic, hypnotic, late-night dancefloor — warm, dubby and groovy",
    "chord_progression": ["Am9", "Fmaj7", "Cmaj7", "Em7"],
    "bars": 16,
    "time_sig": (4, 4),
    "key": "A minor",
}

TRACK_LAYOUT = [
    {"index": 0, "name": "Kick",         "channel": 1,  "pitch_range": [36, 36], "role": "drum_kick"},
    {"index": 1, "name": "Snare/Clap",   "channel": 2,  "pitch_range": [38, 40], "role": "drum_snare_clap"},
    {"index": 2, "name": "Closed Hat",   "channel": 3,  "pitch_range": [42, 42], "role": "drum_chh"},
    {"index": 3, "name": "Open Hat",     "channel": 4,  "pitch_range": [46, 46], "role": "drum_ohh"},
    {"index": 4, "name": "Percussion",   "channel": 5,  "pitch_range": [48, 60], "role": "drum_perc_organic"},
    {"index": 5, "name": "Bass",         "channel": 6,  "pitch_range": [28, 47], "role": "bass"},
    {"index": 6, "name": "Chord/Rhodes", "channel": 7,  "pitch_range": [55, 79], "role": "chord"},
    {"index": 7, "name": "Lead",         "channel": 8,  "pitch_range": [60, 84], "role": "lead_melodic"},
    {"index": 8, "name": "Pad/Texture",  "channel": 9,  "pitch_range": [55, 79], "role": "pad"},
    {"index": 9, "name": "FX/Riser",     "channel": 10, "pitch_range": [60, 96], "role": "fx_riser"},
]

MODEL = "claude-opus-4-7"   # musicality first. for speed/cost use "claude-sonnet-4-6"

# Save config
with open(OUTPUT_DIR / "config.json", "w") as f:
    json.dump({"CONFIG": CONFIG, "TRACK_LAYOUT": TRACK_LAYOUT, "MODEL": MODEL},
              f, indent=2, default=str)
print(f"✓ Config saved")

✓ Config saved


## Cell 3 · 10 Prompt Styles

Even with the same chord progression and genre, the musical character changes completely depending on the style. You can throw a different style every 16 bars, or cycle through them in a fixed order.

In [3]:
PROMPT_STYLES = {
    "classic_groove": {
        "label": "Classic locked groove",
        "rules": [
            "Lock the main groove tightly in bars 1-14. Drums + bass + chords interlock.",
            "Bars 15-16: drum fill (snare/perc roll, hi-hat build) + FX riser tension.",
            "All 10 tracks active at normal density.",
        ],
    },
    "sparse_minimal": {
        "label": "Sparse minimal",
        "rules": [
            "Use HALF the typical note count per track. Embrace negative space.",
            "Kick only on beats 1 and 3 of each bar. Snare/clap only on 2 and 4.",
            "Bass: chord roots only, half-note durations or longer.",
            "Chord: hit only on downbeat of each 4-bar section, long sustain.",
            "Lead/pad: occasional single notes, mostly silent.",
            "Bars 15-16: subtle variation only, don't break the minimalism.",
        ],
    },
    "polyrhythmic": {
        "label": "Polyrhythmic cross-feels",
        "rules": [
            "Drums in straight 4/4 as foundation.",
            "Bass in 3-against-4: dotted-quarter rhythm, cycle repeats every 3 beats.",
            "Lead in 5-against-4 or 7-against-8 feel.",
            "Pad sustains across bar lines (6-beat or 7-beat note durations).",
            "Everything aligns and resolves at bar 16.",
        ],
    },
    "melodic_lead": {
        "label": "Melodic / lead-forward",
        "rules": [
            "LEAD is primary — write a memorable phrase that develops across 16 bars.",
            "Motivic development: state 2-bar idea, repeat with variation, transform.",
            "Drums sparse and supportive — kick + light hats only.",
            "Bass walks under the lead with passing tones.",
            "Chord/pad: harmonic cushion, long sustained notes.",
            "Bars 13-16: melodic climax, leap to highest register, end with resolution.",
        ],
    },
    "bass_driven": {
        "label": "Bass-driven groove",
        "rules": [
            "BASS is the hero — syncopated, prominent, dense 16th-note motion.",
            "Use chromatic passing tones, ghost notes, slides between chord roots.",
            "Drums lock tightly to bass — kick reinforces bass attacks.",
            "Chord: stab only on syncopated off-beats supporting bass groove.",
            "Lead/pad: minimal, textural only.",
            "Bass spans wide register including upper register passages.",
        ],
    },
    "breakbeat": {
        "label": "Broken / breakbeat drums",
        "rules": [
            "Drums BROKEN, not four-on-the-floor. Kick scattered on syncopated 16ths.",
            "Snare hits on unexpected 16th positions; ghost snares throughout.",
            "Hi-hats: fast 16ths/32nds with velocity variation, occasional opens.",
            "Bass: deep sub bombs, sparse but heavy.",
            "Lead/chord: minimal, atmospheric punctuation.",
            "Feel: jungle/DnB energy even at session tempo.",
        ],
    },
    "ambient_pad": {
        "label": "Ambient / atmospheric",
        "rules": [
            "Focus on harmony and texture, not rhythm.",
            "Pad/chord: long sustained notes, 2-4 bar durations each.",
            "Lead: ethereal, sparse single notes with long durations.",
            "Drums: kick only on bar 1 of each 4-bar section, nothing else.",
            "Bass: drone-like, sustained low notes.",
            "Use velocity swells — soft attacks building over 4-8 bars.",
            "Bars 15-16: fade/sustain, no fill.",
        ],
    },
    "build_up": {
        "label": "Building tension (4-bar arc)",
        "rules": [
            "Bars 1-4: SPARSE — just kick + bass root.",
            "Bars 5-8: ADD chord + hats.",
            "Bars 9-12: ADD lead + perc + open hats.",
            "Bars 13-16: FULL DENSITY + climactic fill in last 2 bars.",
            "Velocity ramps: start 70, end 127.",
            "FX/riser builds steadily across all 16 bars.",
        ],
    },
    "call_response": {
        "label": "Call-and-response",
        "rules": [
            "Pattern: every 2 bars alternates between CALL (bars 1-2, 5-6, 9-10, 13-14) and RESPONSE (bars 3-4, 7-8, 11-12, 15-16).",
            "CALL: drums + bass dominant — kick, snare, bass riff statement.",
            "RESPONSE: chord + lead answer the call — drums recede.",
            "Each call/response pair is musically connected (response answers the call).",
            "Intensity builds across the four pairs — final pair most intense.",
        ],
    },
    "glitch_micro": {
        "label": "Glitch / micro-edit",
        "rules": [
            "MANY short notes — 16th and 32nd notes dominate.",
            "Stutters and retriggers: repeat same pitch 3-6 times rapidly.",
            "Pitch jumps within tracks — lead leaps octaves erratically.",
            "Drum hits very tight, occasional 32nd-note rolls.",
            "Velocity highly variable — alternate accents (110+) and ghost notes (40-).",
            "Bass: glitchy 16th-note line with rests/stutters, not legato.",
            "Bars 15-16: extreme stutter or drop — last 2 beats silent or 1-shot fx.",
        ],
    },
}

print(f"✓ {len(PROMPT_STYLES)} prompt styles loaded:")
for k, v in PROMPT_STYLES.items():
    print(f"  {k:20} → {v['label']}")

✓ 10 prompt styles loaded:
  classic_groove       → Classic locked groove
  sparse_minimal       → Sparse minimal
  polyrhythmic         → Polyrhythmic cross-feels
  melodic_lead         → Melodic / lead-forward
  bass_driven          → Bass-driven groove
  breakbeat            → Broken / breakbeat drums
  ambient_pad          → Ambient / atmospheric
  build_up             → Building tension (4-bar arc)
  call_response        → Call-and-response
  glitch_micro         → Glitch / micro-edit


## Cell 4 · Claude API Generator (Streaming)

- `max_tokens=32000` to avoid truncation → forces a streaming call
- Prints the error-position context if JSON parsing fails
- The raw response is auto-saved to `outputs/.../last_raw_response.txt` (for debugging)
- The `style` parameter picks one of PROMPT_STYLES. None = random

In [4]:
SYSTEM_PROMPT = """You are an expert MIDI composer specializing in electronic dance music.
You write groove-driven, harmonically coherent multi-track sequences with musical fills.
You ONLY output valid JSON matching the requested schema — no prose, no markdown fences, no commentary."""

def build_user_prompt(config: dict, track_layout: list, style: str = "classic_groove") -> str:
    style_info = PROMPT_STYLES.get(style, PROMPT_STYLES["classic_groove"])
    style_rules = "\n".join(f"- {r}" for r in style_info["rules"])
    
    track_desc = "\n".join([
        f"  Track {t['index']} — {t['name']}: MIDI channel {t['channel']}, "
        f"pitch {t['pitch_range'][0]}-{t['pitch_range'][1]}, role={t['role']}"
        for t in track_layout
    ])
    bars = config["bars"]
    beats_per_bar = config["time_sig"][0]
    total_beats = bars * beats_per_bar
    chord_lines = "\n".join([
        f"    bars {i*4+1}-{i*4+4} (beats {i*16}-{i*16+15.99}): {ch}"
        for i, ch in enumerate(config["chord_progression"])
    ])
    return f"""Compose {bars} bars in {beats_per_bar}/{config['time_sig'][1]} time.

Tempo: {config['bpm']} BPM
Key: {config['key']}
Genre: {config['genre']}
Mood: {config['mood']}
Style: {style_info['label']}
Total length: {bars} bars = {total_beats} beats (beat index 0..{total_beats}).

Chord progression (4 bars each):
{chord_lines}

Track layout (exactly 10 tracks, output ALL of them in order):
{track_desc}

STYLE-SPECIFIC RULES (most important — follow strictly):
{style_rules}

GENERAL RULES (always apply):
- SEAMLESS LOOP: this block loops continuously in a live DJ set. Bar 1 must land immediately in a full, danceable groove (kick + bass + hats locked) — NO slow fade-in unless the style explicitly says to build up. Make the end wrap back to bar 1 cleanly so the loop point is inaudible.
- TRANSITION: reserve only the final 1-2 bars for a short transition fill / riser that leads into the next block; do not break the groove before then.
- DANCEABILITY: prioritize a driving four-on-the-floor house feel — strong kick on every beat (unless the style says otherwise), syncopated bass, crisp off-beat open/closed hats, and clear forward momentum for the dancefloor.
- Use velocity dynamics (1-127). Accent beats 1 and 3; ghost notes 40-70.
- Durations in beats: 0.0625=32nd, 0.125=16th, 0.25=8th, 0.5=quarter, 1.0=half, 2.0=whole.
- start_beat is GLOBAL position from 0.0 to {total_beats}.
- Stay within each track's pitch range.
- Output ALL 10 tracks even if some have very few notes.

Output ONLY this JSON object (no markdown):
{{
  "meta": {{"bpm": {config['bpm']}, "bars": {bars}, "key": "{config['key']}", "style": "{style}"}},
  "tracks": [
    {{"track_index": 0, "track_name": "Kick", "notes": [{{"pitch": 36, "start_beat": 0.0, "duration": 0.25, "velocity": 110}}]}}
  ]
}}
"""

def generate_midi_with_claude(config: dict, track_layout: list,
                              model: str = None, style: str = None,
                              max_tokens: int = 32000, save_raw: bool = True) -> dict:
    """Claude API (streaming) -> 10-track, 16-bar MIDI dict.
    style=None selects a random style.
    """
    model = model or MODEL
    if style is None:
        style = random.choice(list(PROMPT_STYLES.keys()))
    style_info = PROMPT_STYLES[style]
    print(f"[gen] style={style} ({style_info['label']}) model={model}")
    
    user_prompt = build_user_prompt(config, track_layout, style=style)

    # Streaming call -- required when max_tokens is large
    chunks = []
    t0 = time.time()
    print("      streaming…", end="", flush=True)
    with client.messages.stream(
        model=model,
        max_tokens=max_tokens,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_prompt}],
    ) as stream:
        for text in stream.text_stream:
            chunks.append(text)
        final = stream.get_final_message()
    raw = "".join(chunks).strip()
    print(f" {time.time()-t0:.1f}s, {len(raw)} chars, stop={final.stop_reason}")

    # Save raw response
    debug_path = OUTPUT_DIR / "last_raw_response.txt"
    if save_raw:
        with open(debug_path, "w") as f:
            f.write(raw)

    if final.stop_reason == "max_tokens":
        print(f"⚠ Response truncated at max_tokens={max_tokens}")

    # Strip code fences
    if "```" in raw:
        for p in raw.split("```"):
            p = p.strip()
            if p.startswith("json"): p = p[4:].strip()
            if p.startswith("{"):
                raw = p
                break

    # Extract the JSON body
    start = raw.find("{")
    end = raw.rfind("}")
    if start != -1 and end != -1 and end > start:
        raw = raw[start:end + 1]

    # Parse + error context
    try:
        data = json.loads(raw)
    except json.JSONDecodeError as e:
        a, b = max(0, e.pos - 200), min(len(raw), e.pos + 200)
        print(f"\n⚠ JSONDecodeError at char {e.pos}: {e.msg}")
        print(f"\n--- 200 chars BEFORE ---\n{raw[a:e.pos]}")
        print(f"\n>>> ERROR <<<\n")
        print(f"--- 200 chars AFTER ---\n{raw[e.pos:b]}")
        print(f"\nFull response: {debug_path}")
        raise

    assert "tracks" in data and len(data["tracks"]) == 10, \
        f"expected 10 tracks, got {len(data.get('tracks', []))}"
    return data

print("✓ Claude API generator ready")

✓ Claude API generator ready


## Cell 5 · MIDI Output Port (IAC Driver)

List the available output ports → set `MIDI_PORT_NAME` to the exact string → open it.

In [5]:
print("Available MIDI output ports:")
for i, p in enumerate(mido.get_output_names()):
    print(f"  [{i}] {p}")

# Edit to match your setup (the exact string from the list above)
MIDI_PORT_NAME = "IAC Driver Bus 1"   # macOS
# MIDI_PORT_NAME = "loopMIDI Port"     # Windows

# Close a previously opened port if any (safe to re-run)
try:
    if midi_out is not None:
        midi_out.close()
except Exception:
    pass

midi_out = mido.open_output(MIDI_PORT_NAME)
print(f"\n✓ Opened: {MIDI_PORT_NAME}")

Available MIDI output ports:
  [0] IAC Driver Bus 1

✓ Opened: IAC Driver Bus 1


## Cell 6 · AbletonOSC (optional — track colors + automatic Session View clip saving)

**Run only if needed.** The AbletonOSC device must be enabled as a Control Surface.
The sequencer works fine without OSC — track colors simply fall back to white and automatic Session View saving is disabled.

In [6]:
# Clean up any previous OSC servers (safe to re-run / includes orphaned threads)
import gc, socketserver
_closed = 0
for _obj in list(gc.get_objects()):
    if isinstance(_obj, socketserver.BaseServer):
        try:
            _obj.shutdown()       # stop serve_forever loop (returns immediately if not running)
            _obj.server_close()   # release socket -> free the port
            _closed += 1
        except Exception:
            pass
if _closed:
    print(f"(cleaned up {_closed} previous OSC server(s))")

if HAS_OSC:
    ABLETON_HOST       = "127.0.0.1"
    ABLETON_RECV_PORT  = 11000   # AbletonOSC receive
    LOCAL_RECV_PORT    = 11001   # response receive

    osc_client = udp_client.SimpleUDPClient(ABLETON_HOST, ABLETON_RECV_PORT)

    class LiveQuery:
        """Helper that sends an OSC request and receives the response synchronously."""
        def __init__(self):
            self._responses = {}
            self._events = {}
            self._lock = threading.Lock()

        def _handler(self, addr, *args):
            with self._lock:
                key = args[0] if args else None
                value = args[1:] if len(args) > 1 else args
                self._responses.setdefault(addr, {})[key] = value
                ev = self._events.get((addr, key))
                if ev: ev.set()

        def request(self, addr, params=None, key=None, timeout=1.0):
            with self._lock:
                ev = threading.Event()
                self._events[(addr, key)] = ev
            osc_client.send_message(addr, params or [])
            if not ev.wait(timeout): return None
            with self._lock:
                val = self._responses[addr].get(key)
            return val[0] if val and len(val) == 1 else val

    lq = LiveQuery()
    _disp = dispatcher.Dispatcher()
    _disp.set_default_handler(lq._handler)

    class _ReusableOSCServer(osc_server.ThreadingOSCUDPServer):
        allow_reuse_address = True

    # Reclaim the reply port from any other process (e.g. a stale kernel) before binding.
    # Authorized hard-kill: whatever holds LOCAL_RECV_PORT gets SIGKILL so OSC always connects.
    import subprocess as _sp, os as _os, signal as _sig
    try:
        _holders = {int(x) for x in _sp.run(
            ["lsof", "-nP", f"-tiUDP:{LOCAL_RECV_PORT}"],
            capture_output=True, text=True).stdout.split()} - {_os.getpid()}
        for _pid in _holders:
            try:
                _os.kill(_pid, _sig.SIGKILL)
                print(f"  (freed port {LOCAL_RECV_PORT}: killed PID {_pid})")
            except Exception as _e:
                print(f"  (could not kill PID {_pid}: {_e})")
        if _holders:
            time.sleep(0.3)
    except Exception:
        pass

    _osc_server = _ReusableOSCServer((ABLETON_HOST, LOCAL_RECV_PORT), _disp)
    threading.Thread(target=_osc_server.serve_forever, daemon=True).start()
    print(f"✓ OSC: send→{ABLETON_HOST}:{ABLETON_RECV_PORT}  recv←{ABLETON_HOST}:{LOCAL_RECV_PORT}")

    # Connection test
    n = lq.request("/live/song/get/num_tracks", key=0, timeout=1.0)
    if n is None:
        print("! no AbletonOSC response -- check that AbletonOSC is selected as a Control Surface")
        # is another process (e.g. a stale Jupyter kernel) holding our reply port?
        import subprocess as _sp, os as _os
        try:
            _out = _sp.run(["lsof", "-nP", f"-iUDP:{LOCAL_RECV_PORT}"],
                           capture_output=True, text=True).stdout
            _pids = sorted({l.split()[1] for l in _out.splitlines()[1:]
                            if l.split()[1] != str(_os.getpid())})
            if _pids:
                print(f"  -> port {LOCAL_RECV_PORT} is ALSO held by PID(s) {_pids} "
                      f"(likely a stale kernel stealing the replies).")
                print(f"     Stop them and re-run this cell:  kill {' '.join(_pids)}")
        except Exception:
            pass
        lq = None  # disable
    else:
        print(f"   Ableton tracks: {n}")
        # push the notebook tempo to Ableton so they always match
        osc_client.send_message("/live/song/set/tempo", [float(CONFIG["bpm"])])
        print(f"   set Ableton tempo -> {CONFIG['bpm']} BPM")
else:
    print("OSC section skipped (python-osc not installed)")


✓ OSC: send→127.0.0.1:11000  recv←127.0.0.1:11001
! no AbletonOSC response -- check that AbletonOSC is selected as a Control Surface


In [7]:
# Track color / name query helpers
def color_int_to_hex(c):
    return f"#{int(c) & 0xFFFFFF:06X}" if c is not None else None

def get_track_color(track_idx, timeout=1.0):
    if lq is None: return None
    val = lq.request("/live/track/get/color", [track_idx], key=track_idx, timeout=timeout)
    if val is None: return None
    if isinstance(val, (tuple, list)) and len(val) >= 1: return int(val[0])
    return int(val)

def get_track_name(track_idx, timeout=1.0):
    if lq is None: return None
    val = lq.request("/live/track/get/name", [track_idx], key=track_idx, timeout=timeout)
    if val is None: return None
    if isinstance(val, (tuple, list)) and len(val) >= 1: return str(val[0])
    return str(val)

def fetch_all_track_colors(n=10):
    """Return a list of #RRGGBB colors for n tracks. Falls back to #FFFFFF on failure."""
    if lq is None: return None
    return [color_int_to_hex(get_track_color(i, timeout=0.3)) or "#FFFFFF" for i in range(n)]

# Show current colors
if lq is not None:
    for i in range(len(TRACK_LAYOUT)):
        c = get_track_color(i)
        nm = get_track_name(i)
        print(f"  Track {i:>2} [{nm or '?':<24}]  {color_int_to_hex(c) or '------'}")

---
## Cell 6.5 · Synth Scanner (auto-detect Analog / Operator)

When OSC is enabled, this auto-detects the Analog/Operator devices in the current session and selects their tweakable parameters.

In [8]:
from dataclasses import dataclass, field as dc_field
from copy import deepcopy

# -- Add a direct-query layer on top of the existing dispatcher --
_pending_queries: dict = {}
_pending_lock = threading.Lock()

def _multi_handler(addr, *args):
    """Dispatcher handler that serves both lq and direct queries."""
    if lq is not None:
        lq._handler(addr, *args)
    with _pending_lock:
        if addr in _pending_queries:
            ev, holder = _pending_queries[addr]
            holder["args"] = args
            ev.set()

if lq is not None:
    _disp.set_default_handler(_multi_handler)
    print("OK  dispatcher patched")

def synth_query(address, *args, timeout=2.0):
    """Send an OSC request and receive the response as a raw args tuple."""
    if osc_client is None:
        return None
    ev = threading.Event()
    holder: dict = {}
    with _pending_lock:
        _pending_queries[address] = (ev, holder)
    osc_client.send_message(address, list(args))
    ev.wait(timeout=timeout)
    with _pending_lock:
        _pending_queries.pop(address, None)
    return holder.get("args")

# -- SynthTarget dataclass --
@dataclass
class SynthTarget:
    name: str
    track_id: int
    device_id: int
    param_names:     list = dc_field(default_factory=list)
    param_values:    list = dc_field(default_factory=list)
    param_mins:      list = dc_field(default_factory=list)
    param_maxs:      list = dc_field(default_factory=list)
    param_quantized: list = dc_field(default_factory=list)

# -- Track scan: detect Analog / Operator --
synths: dict = {}
SYNTH_TARGET_CLASSES = {"Analog", "Operator"}
SYNTH_EXCLUDE_KW = [
    "on/off","device on","volume","output","level","gain","pan",
    "mode","type","routing","algorithm","voices","polyphony",
    "retrigger","glide","legato","spread","time","sync",
]

def fetch_device_params(st: SynthTarget):
    for attr, endpoint in [
        ("param_names",     "/live/device/get/parameters/name"),
        ("param_values",    "/live/device/get/parameters/value"),
        ("param_mins",      "/live/device/get/parameters/min"),
        ("param_maxs",      "/live/device/get/parameters/max"),
        ("param_quantized", "/live/device/get/parameters/is_quantized"),
    ]:
        resp = synth_query(endpoint, st.track_id, st.device_id, timeout=3.0)
        if not resp:
            continue
        raw = list(resp[2:])  # drop the first two (track_id, device_id)
        setattr(st, attr, [bool(q) for q in raw] if attr == "param_quantized" else raw)

def select_tweakable(st: SynthTarget):
    out = []
    for i, name in enumerate(st.param_names):
        low = name.lower()
        if any(kw in low for kw in SYNTH_EXCLUDE_KW): continue
        if i < len(st.param_quantized) and st.param_quantized[i]: continue
        if i < len(st.param_mins) and st.param_mins[i] >= st.param_maxs[i]: continue
        out.append(i)
    return out

if lq is not None:
    nt_resp = synth_query("/live/song/get/num_tracks", timeout=2.0)
    num_synth_tracks = int(nt_resp[0]) if nt_resp else 0
    print(f"Total tracks: {num_synth_tracks}")

    for _t in range(num_synth_tracks):
        time.sleep(0.05)
        _resp = synth_query("/live/track/get/devices/class_name", _t, timeout=1.5)
        if not _resp: continue
        for _d, _cls in enumerate(_resp[1:]):
            if _cls in SYNTH_TARGET_CLASSES and _cls not in synths:
                synths[_cls] = SynthTarget(name=_cls, track_id=_t, device_id=_d)
                print(f"  🎹 {_cls} → Track {_t}, Device {_d}")

    synth_tweakable: dict = {}
    for _name, _st in synths.items():
        fetch_device_params(_st)
        synth_tweakable[_name] = select_tweakable(_st)
        print(f"  🎛️  {_name}: {len(synth_tweakable[_name])}/{len(_st.param_names)} params tweakable")

    if not synths:
        print("!  No Analog/Operator found -- continuing without synth control.")
else:
    synth_tweakable = {}
    print("!  OSC disabled -- skipping synth scan")

!  OSC disabled -- skipping synth scan


---
## Cell 6.6 · SynthManager — timbre evolution engine

While each 16-bar block plays, the Analog/Operator parameters are linearly interpolated from their current values toward target values.
On each block change a new drift target is generated so the timbre mutates naturally.

In [9]:
def _rand_target(st, indices: list, range_pct=1.0) -> dict:
    out = {}
    for idx in indices:
        mn, mx = st.param_mins[idx], st.param_maxs[idx]
        center = (mn + mx) / 2.0
        half   = (mx - mn) * range_pct / 2.0
        out[idx] = random.uniform(max(mn, center - half), min(mx, center + half))
    return out

def _drift_target(st, indices: list, current: dict, drift_pct=0.15) -> dict:
    out = {}
    for idx in indices:
        mn, mx   = st.param_mins[idx], st.param_maxs[idx]
        drift    = (mx - mn) * drift_pct
        cur      = current.get(idx, st.param_values[idx])
        out[idx] = random.uniform(max(mn, cur - drift), min(mx, cur + drift))
    return out


class SynthManager:
    """Gradually morphs the Analog/Operator timbre by interpolation during block playback."""

    def __init__(self, osc_client_ref, synths_dict: dict, tweakable_dict: dict):
        self.client    = osc_client_ref
        self.synths    = synths_dict
        self.tweakable = tweakable_dict
        self._cur: dict = {}
        self._tgt: dict = {}
        self._stop  = threading.Event()
        self._thread = None

    def _apply(self, st, params: dict):
        for idx, val in params.items():
            self.client.send_message(
                "/live/device/set/parameter/value",
                [st.track_id, st.device_id, idx, float(val)],
            )

    def initialize(self, range_pct=0.8, drift_pct=0.15):
        for name, st in self.synths.items():
            idx = self.tweakable.get(name, [])
            if not idx: continue
            init = _rand_target(st, idx, range_pct)
            self._apply(st, init)
            self._cur[name] = init
            self._tgt[name] = _drift_target(st, idx, init, drift_pct)
        print(f"  *  applied initial synth timbre ({', '.join(self.synths.keys())})")

    def start_block_evolution(self, block_sec: float, drift_pct=0.15, poll_interval=0.08):
        self.stop()
        for name, st in self.synths.items():
            if name in self._tgt:
                self._cur[name] = deepcopy(self._tgt[name])
            idx = self.tweakable.get(name, [])
            if idx:
                self._tgt[name] = _drift_target(st, idx, self._cur.get(name, {}), drift_pct)

        self._stop.clear()
        start = time.monotonic()

        def _run():
            while not self._stop.is_set():
                elapsed = time.monotonic() - start
                t = min(1.0, elapsed / block_sec)
                for name, st in self.synths.items():
                    interp = {}
                    for idx in self.tweakable.get(name, []):
                        a = self._cur.get(name, {}).get(idx, st.param_values[idx])
                        b = self._tgt.get(name, {}).get(idx, a)
                        interp[idx] = a + (b - a) * t
                    if interp:
                        self._apply(st, interp)
                if t >= 1.0:
                    break
                time.sleep(poll_interval)

        self._thread = threading.Thread(target=_run, daemon=True, name="synth_evo")
        self._thread.start()

    def stop(self):
        self._stop.set()
        if self._thread and self._thread.is_alive():
            self._thread.join(timeout=1.0)
        self._thread = None


# -- Create instance --
if synths and osc_client is not None:
    synth_manager = SynthManager(osc_client, synths, synth_tweakable)
    print("OK  SynthManager ready")
else:
    synth_manager = None
    print("i  No SynthManager (no synths found or OSC disabled)")


i  No SynthManager (no synths found or OSC disabled)


---
## Cell 7 · ClipLooper & Piano Roll

Defines `ClipLooper` (loop-plays Session View clips) and `plot_piano_roll` (renders an interactive HTML/SVG piano roll).

In [10]:
class ClipLooper:
    """Fires Session View clips in order and waits for each block to finish."""

    def __init__(self, osc_client_ref, config: dict, n_tracks: int = 10):
        self.osc         = osc_client_ref
        self.block_beats = config["bars"] * config["time_sig"][0]
        self.beat_sec    = 60.0 / config["bpm"]
        self.block_sec   = self.block_beats * self.beat_sec
        self.n_tracks    = n_tracks
        self.stop_event  = threading.Event()
        self._fire_time: float = 0.0

    def fire_slot(self, slot_idx: int):
        """Fire every track clip in the given slot (column) at once."""
        if self.osc is None:
            return
        for ti in range(self.n_tracks):
            self.osc.send_message("/live/clip_slot/fire", [ti, slot_idx])
            time.sleep(0.004)
        self._fire_time = time.monotonic()
        print(f"  -> clip fire: slot {slot_idx + 1}")

    def wait_for_loop_end(self, switch_ahead: float = 0.12):
        """Block until switch_ahead seconds before the current block ends."""
        target = self.block_sec - switch_ahead
        while not self.stop_event.is_set():
            elapsed = time.monotonic() - self._fire_time
            if elapsed >= target:
                return
            remaining = target - elapsed
            time.sleep(min(0.02, remaining * 0.5))

    def elapsed(self) -> float:
        return time.monotonic() - self._fire_time

    def stop_all_clips(self):
        if self.osc:
            self.osc.send_message("/live/song/stop_all_clips", [])


def build_piano_roll_html(midi_data, config, track_layout, track_colors=None):
    """Return a standalone, monochrome, interactive piano-roll HTML document.
    track_colors is ignored on purpose: the scheme is black & white, with note
    brightness mapped to velocity."""
    bars        = config["bars"]
    bpb         = config["time_sig"][0]
    total_beats = bars * bpb
    n_tracks    = len(track_layout)

    PPB, LEFT, PADX, LANE, GAP = 16, 96, 10, 58, 10
    W = LEFT + total_beats * PPB + PADX
    H = n_tracks * (LANE + GAP)

    def gray(vel):
        g = int(70 + (max(1, min(127, int(vel))) / 127.0) * 175)   # 70..245
        return f"#{g:02x}{g:02x}{g:02x}"

    lanes = []
    for i, track in enumerate(midi_data["tracks"]):
        meta    = track_layout[i]
        y0      = i * (LANE + GAP)
        notes   = track["notes"]
        pitches = [nn["pitch"] for nn in notes]
        lo, hi  = (min(pitches), max(pitches)) if pitches else tuple(meta["pitch_range"])
        if hi <= lo:
            hi = lo + 1
        span  = hi - lo
        noteH = max(3.0, (LANE - 6) / (span + 1))

        p = [f'<rect x="{LEFT}" y="{y0}" width="{total_beats*PPB}" height="{LANE}" class="lane"/>']
        fx = LEFT + (bars - 2) * bpb * PPB
        p.append(f'<rect x="{fx}" y="{y0}" width="{2*bpb*PPB}" height="{LANE}" class="fill"/>')
        for b in range(bars + 1):
            x = LEFT + b * bpb * PPB
            p.append(f'<line x1="{x}" y1="{y0}" x2="{x}" y2="{y0+LANE}" class="bar"/>')
        for b in range(total_beats + 1):
            if b % bpb == 0:
                continue
            x = LEFT + b * PPB
            p.append(f'<line x1="{x}" y1="{y0}" x2="{x}" y2="{y0+LANE}" class="beat"/>')
        p.append(f'<text x="{LEFT-8}" y="{y0+LANE/2:.1f}" class="lbl">{meta["name"]}</text>')
        for nn in notes:
            sb   = float(nn["start_beat"])
            dur  = max(0.08, float(nn["duration"]))
            vel  = int(nn["velocity"])
            pit  = int(nn["pitch"])
            x    = LEFT + sb * PPB
            w    = max(2.0, dur * PPB - 0.6)
            frac = (pit - lo) / span
            y    = y0 + (LANE - noteH) - frac * (LANE - noteH - 3)
            p.append(
                f'<rect class="note" x="{x:.1f}" y="{y:.1f}" width="{w:.1f}" height="{noteH:.1f}" '
                f'fill="{gray(vel)}" data-i="pitch {pit} | vel {vel} | @{sb:g} | {dur:g}b"/>'
            )
        lanes.append("".join(p))

    style_tag = midi_data.get("meta", {}).get("style", "")
    header = (f"{config['genre']} &middot; {config['mood']} &middot; style={style_tag}<br>"
              f"{' &rarr; '.join(config['chord_progression'])} &middot; "
              f"{config['bpm']} BPM &middot; {config['key']}")
    svg = (f'<svg viewBox="0 0 {W} {H}" width="100%" preserveAspectRatio="xMidYMid meet">'
           f'{"".join(lanes)}</svg>')

    return f"""<!DOCTYPE html><html lang="en"><head><meta charset="UTF-8"><style>
*{{margin:0;padding:0;box-sizing:border-box}}
body{{background:#0f0f0f;color:#ececec;font-family:-apple-system,Segoe UI,Roboto,sans-serif;padding:10px 14px}}
.hd{{font-size:12px;color:#b8b8b8;line-height:1.5;margin-bottom:8px;font-family:ui-monospace,Menlo,monospace}}
svg text{{font-family:ui-monospace,Menlo,monospace}}
.lane{{fill:#1b1b1b}}
.fill{{fill:#ffffff;opacity:.05}}
.bar{{stroke:#555;stroke-width:1}}
.beat{{stroke:#2c2c2c;stroke-width:.5}}
.lbl{{fill:#cfcfcf;font-size:11px;text-anchor:end;dominant-baseline:middle}}
.note{{stroke:#0f0f0f;stroke-width:.4;cursor:pointer}}
.note:hover{{stroke:#fff;stroke-width:1.2}}
#tip{{position:fixed;pointer-events:none;background:#000;border:1px solid #666;color:#fff;
font:11px ui-monospace,Menlo,monospace;padding:4px 7px;border-radius:4px;opacity:0;
transition:opacity .1s;white-space:nowrap;z-index:99}}
</style></head><body>
<div class="hd">{header}</div>
{svg}
<div id="tip"></div>
<script>
const tip=document.getElementById('tip');
document.querySelectorAll('.note').forEach(n=>{{
  n.addEventListener('mousemove',e=>{{tip.textContent=n.dataset.i;tip.style.opacity=1;
    tip.style.left=(e.clientX+12)+'px';tip.style.top=(e.clientY+12)+'px';}});
  n.addEventListener('mouseleave',()=>{{tip.style.opacity=0}});
}});
</script></body></html>"""


def plot_piano_roll(midi_data, config, track_layout, save_path, track_colors=None):
    """Build an interactive monochrome HTML piano roll, save it to save_path
    (extension forced to .html) and, when run on the main thread, display it
    inline in the notebook."""
    html = build_piano_roll_html(midi_data, config, track_layout, track_colors)
    save_path = Path(save_path)
    if save_path.suffix.lower() != ".html":
        save_path = save_path.with_suffix(".html")
    save_path.write_text(html, encoding="utf-8")

    if threading.current_thread() is threading.main_thread():
        try:
            from IPython.display import HTML, display
            h = 70 + len(track_layout) * 68
            srcdoc = html.replace('"', "&quot;")
            display(HTML(
                f'<iframe srcdoc="{srcdoc}" style="width:100%;height:{h}px;'
                f'border:1px solid #2c2c2c;border-radius:8px;background:#0f0f0f"></iframe>'
            ))
        except Exception:
            pass
    return save_path

print("OK  ClipLooper + interactive HTML piano roll ready")


OK  ClipLooper + interactive HTML piano roll ready


## Cell 8 · Session View Clip Writer

Saves each 16-bar block as 10 clips into a single column of the Session View.
Uses AbletonOSC's `/live/clip/add/notes` endpoint (the official spec).

In [11]:
def write_block_to_session(midi_data, slot_index, config, track_layout,
                            overwrite=True, verbose=False):
    """Save a 16-bar block as 10 clips into Session View column `slot_index` (0-indexed).
    osc_client must be active."""
    if osc_client is None or lq is None:
        if verbose: print("! OSC disabled -- skipping Session save")
        return
    
    bars = config["bars"]
    length_beats = float(bars * config["time_sig"][0])
    
    # Clip name: e.g. "01 · Am9 · deep house"
    first_chord = config["chord_progression"][0]
    genre_short = " ".join(config["genre"].split()[:2])
    style_tag = midi_data.get("meta", {}).get("style", "")
    clip_name = f"{slot_index+1:02d} · {first_chord}"
    if style_tag:
        clip_name += f" · {style_tag}"
    clip_name = clip_name[:40]
    
    for track in midi_data["tracks"]:
        ti = track["track_index"]
        
        # 1) Delete existing clip (overwrite)
        if overwrite:
            try:
                osc_client.send_message("/live/clip_slot/delete_clip", [ti, slot_index])
                time.sleep(0.02)
            except Exception:
                pass
        
        # 2) Create empty MIDI clip
        osc_client.send_message("/live/clip_slot/create_clip",
                                [ti, slot_index, length_beats])
        time.sleep(0.04)
        
        # 3) Set clip name
        osc_client.send_message("/live/clip/set/name",
                                [ti, slot_index, clip_name])
        time.sleep(0.02)
        
        # 4) Add notes -- /live/clip/add/notes (official endpoint)
        notes_in_range = [n for n in track["notes"]
                          if 0 <= n["start_beat"] < length_beats]
        if notes_in_range:
            lo, hi = track_layout[ti]["pitch_range"]
            args = [ti, slot_index]
            for n in notes_in_range:
                pitch = int(max(lo, min(hi, n["pitch"])))
                start = float(n["start_beat"])
                dur   = max(0.0625, float(n["duration"]))
                vel   = int(max(1, min(127, n["velocity"])))
                args.extend([pitch, start, dur, vel, False])  # mute=False (boolean)
            osc_client.send_message("/live/clip/add/notes", args)
            time.sleep(0.03)
        
        if verbose:
            print(f"   track {ti}: {len(notes_in_range)} notes → slot {slot_index}")
    
    print(f"✓ Session View column {slot_index+1}: '{clip_name}'")

print("✓ write_block_to_session ready")

✓ write_block_to_session ready


---
## Cell 9 · Live Session (run_live_session)

- Fires each block with ClipLooper → Ableton loop-plays it
- Switches to the next block only after the current loop completes
- Evolves the timbre simultaneously with SynthManager
- Can also be launched from the Start Session button in the web dashboard (Cell 11)

In [12]:
import queue as _queue


def _pick_style(styles, iteration):
    if styles is None:           return None
    if isinstance(styles, str):  return styles
    if isinstance(styles, list): return styles[iteration % len(styles)]
    return None


# ── Camelot wheel (harmonic mixing) ──────────────────────────────────────
# Each entry: camelot number -> (key name, tonic pitch-class, prefer flats)
CAMELOT_MINOR = {
    1:  ("Ab minor", 8,  True),
    2:  ("Eb minor", 3,  True),
    3:  ("Bb minor", 10, True),
    4:  ("F minor",  5,  True),
    5:  ("C minor",  0,  True),
    6:  ("G minor",  7,  True),
    7:  ("D minor",  2,  True),
    8:  ("A minor",  9,  False),
    9:  ("E minor",  4,  False),
    10: ("B minor",  11, False),
    11: ("F# minor", 6,  False),
    12: ("C# minor", 1,  False),
}
_SHARP = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
_FLAT  = ["C", "Db", "D", "Eb", "E", "F", "Gb", "G", "Ab", "A", "Bb", "B"]
# Base progression in A minor (Am9 - Fmaj7 - Cmaj7 - Em7) = i - VI - III - v
_PROG_OFFSETS = [0, 8, 3, 7]
_PROG_QUALITY = ["m9", "maj7", "maj7", "m7"]
_KEY_TO_CAMELOT = {meta[0]: num for num, meta in CAMELOT_MINOR.items()}


def key_to_camelot(key_name: str) -> int:
    """Camelot number for a minor key name (defaults to 8A = A minor)."""
    return _KEY_TO_CAMELOT.get(key_name, 8)


def camelot_progression(num: int):
    """Return (key_name, [4 chord symbols]) for a Camelot minor number.
    Going +1 on the wheel moves up a perfect fifth -> smooth harmonic transition."""
    num = ((num - 1) % 12) + 1
    key_name, tonic, use_flat = CAMELOT_MINOR[num]
    names = _FLAT if use_flat else _SHARP
    chords = [names[(tonic + off) % 12] + q
              for off, q in zip(_PROG_OFFSETS, _PROG_QUALITY)]
    return key_name, chords


def run_live_session(
    config,
    track_layout,
    n_iterations:  int   = None,   # None = endless (until Stop / Interrupt)
    lead_in:       float = 3.0,
    start_slot:    int   = 0,
    styles               = None,
    sm                   = None,   # SynthManager instance (no timbre evolution if None)
    drift_pct:     float = 0.35,
    switch_ahead:  float = 0.12,
    buffer_ahead:  int   = 2,      # how many blocks to keep generated ahead of playback
    slot_ring:     int   = 8,      # reuse this many Session slots in a ring
    rotate_camelot: bool = True,   # step +1 on the Camelot wheel every block
):
    """Seamless live set: a background producer always keeps `buffer_ahead`
    blocks generated and written to Session slots. The conductor fires the next
    block exactly at the loop boundary; if the next block is not ready yet it
    fires NOTHING, so the current clip keeps looping with no audio gap."""
    looper    = ClipLooper(osc_client, config, n_tracks=len(track_layout))
    block_sec = looper.block_sec
    stop      = looper.stop_event

    # publish the looper so the web dashboard Stop button works
    _ref = globals().get("_web_looper_ref")
    if isinstance(_ref, list) and len(_ref) >= 1:
        _ref[0] = looper

    # boundary-aligned, seamless clip launches (1-bar trigger quantization)
    if osc_client is not None:
        try:
            osc_client.send_message("/live/song/set/clip_trigger_quantization", [4])
            osc_client.send_message("/live/song/set/tempo", [float(config["bpm"])])
        except Exception:
            pass

    start_cam = key_to_camelot(config.get("key", "A minor"))

    def block_config(k):
        cfg = {**config, "chord_progression": list(config["chord_progression"])}
        if rotate_camelot:
            cam = ((start_cam - 1 + k) % 12) + 1
            key_name, chords = camelot_progression(cam)
            cfg["key"] = key_name
            cfg["chord_progression"] = chords
            cfg["_camelot"] = f"{cam}A"
        return cfg

    print(f"Block: {config['bars']} bars = {block_sec:.1f}s @ {config['bpm']} BPM")
    print(f"  OSC          : {'ON' if lq else 'OFF'}")
    print(f"  SynthManager : {'ON (' + ', '.join(sm.synths.keys()) + ')' if sm else 'OFF'}")
    print(f"  Styles       : {styles or 'RANDOM'}")
    print(f"  Camelot       : {'ON (start ' + str(start_cam) + 'A, +1/block)' if rotate_camelot else 'OFF'}")
    print(f"  Iterations   : {n_iterations if n_iterations is not None else 'ENDLESS'}")
    print(f"  Buffer ahead : {buffer_ahead} block(s)")

    ready = _queue.Queue()   # items: {slot, data, cfg, idx}; None = producer finished

    # ── Producer: always stays `buffer_ahead` blocks ahead of playback ────
    def _producer():
        k = 0
        while not stop.is_set():
            if n_iterations is not None and k >= n_iterations:
                break
            while not stop.is_set() and ready.qsize() >= buffer_ahead:
                time.sleep(0.1)
            if stop.is_set():
                break
            cfg  = block_config(k)
            slot = start_slot + (k % slot_ring)
            try:
                data = generate_midi_with_claude(cfg, track_layout, style=_pick_style(styles, k))
                write_block_to_session(data, slot, cfg, track_layout)
            except Exception as e:
                print(f"  !! generation failed (block {k+1}): {e} -- retrying")
                time.sleep(1.0)
                continue
            ready.put({"slot": slot, "data": data, "cfg": cfg, "idx": k})
            k += 1
        ready.put(None)

    producer = threading.Thread(target=_producer, daemon=True, name="block_producer")
    producer.start()

    # ── Helpers ───────────────────────────────────────────────────────────
    def _wait_until(t_mono):
        while not stop.is_set():
            now = time.monotonic()
            if now >= t_mono:
                return
            time.sleep(min(0.02, (t_mono - now) * 0.5 + 0.001))

    def _present(item):
        cfg = item["cfg"]
        iter_dir = OUTPUT_DIR / f"iter_{item['idx']+1:03d}"
        iter_dir.mkdir(exist_ok=True)
        with open(iter_dir / "midi.json", "w") as f:
            json.dump(item["data"], f, indent=2)
        threading.Thread(
            target=plot_piano_roll,
            args=(item["data"], cfg, track_layout, iter_dir / "piano_roll.html", None),
            daemon=True,
        ).start()
        notes_total = sum(len(t["notes"]) for t in item["data"]["tracks"])
        style_tag   = item["data"].get("meta", {}).get("style", "?")
        print(f"> block {item['idx']+1}  slot={item['slot']+1}  key={cfg.get('key')}"
              f"  {cfg.get('_camelot','')}  style={style_tag}  notes={notes_total}")

    # ── Wait for the first block, then start ──────────────────────────────
    print("\n[Pre-roll] generating first block(s)...")
    cur = ready.get()
    if cur is None:
        print("no blocks produced."); return
    if sm:
        sm.initialize(range_pct=0.9, drift_pct=drift_pct)
    print(f"\n> starting in {lead_in:.0f}s -- press Play in Ableton.\n")
    time.sleep(lead_in)

    try:
        looper.fire_slot(cur["slot"])
        _present(cur)
        if sm:
            sm.start_block_evolution(block_sec, drift_pct=drift_pct)
        boundary = time.monotonic() + block_sec

        while not stop.is_set():
            _wait_until(boundary - switch_ahead)
            if stop.is_set():
                break
            boundary += block_sec

            # next block ready? advance and fire it; otherwise keep looping current
            try:
                nxt = ready.get_nowait()
            except _queue.Empty:
                nxt = "WAIT"

            if isinstance(nxt, dict):
                cur = nxt
                looper.fire_slot(cur["slot"])
                _present(cur)
            elif nxt is None:
                # producer finished — keep looping the final block seamlessly
                pass
            # nxt == "WAIT": not ready yet — fire nothing, current clip keeps looping

            # re-target the synth every boundary -> continuous, pronounced evolution
            if sm:
                sm.start_block_evolution(block_sec, drift_pct=drift_pct)

    except KeyboardInterrupt:
        print("\n[stopped]")
    finally:
        stop.set()
        if sm:
            sm.stop()
        looper.stop_all_clips()
        print(f"\nOK done. Outputs: {OUTPUT_DIR.resolve()}")

print("OK  run_live_session ready (seamless buffered playback + Camelot rotation)")


OK  run_live_session ready (seamless buffered playback + Camelot rotation)


---
## Cell 10 · Dry Run (generate + visualize, no playback)

Check the Claude call + JSON parsing + piano roll. For testing before starting a session.

In [13]:
# style=None -> random; or a specific PROMPT_STYLES key (e.g. "build_up")
test_data = generate_midi_with_claude(CONFIG, TRACK_LAYOUT, style=None)

test_dir = OUTPUT_DIR / "dry_run"
test_dir.mkdir(exist_ok=True)
with open(test_dir / "midi.json", "w") as f:
    json.dump(test_data, f, indent=2)

_colors = fetch_all_track_colors(len(TRACK_LAYOUT))
plot_piano_roll(test_data, CONFIG, TRACK_LAYOUT, test_dir / "piano_roll.html",
                track_colors=_colors)

print("\nNote counts:")
for t in test_data["tracks"]:
    print(f"  Track {t['track_index']:>2} {t.get('track_name', '?'):<14}  notes={len(t['notes']):>3}")


[gen] style=glitch_micro (Glitch / micro-edit) model=claude-opus-4-7
      streaming…

KeyboardInterrupt: 

---
## Cell 11 · 🌐 Web Dashboard

Runs a Flask server in the background → open **http://127.0.0.1:5056** in your browser.

| Section | Function |
|---|---|
| ① Config (left 272px) | BPM · Key · Genre · Mood · Chord · Style selection · Drift · start/stop session |
| ② Piano Roll (center, wide) | Live interactive HTML piano roll on every block · slot tabs · ← → navigation · HTML download |
| ③ MIDI Director (right 316px) | Instruct Claude for the next block → extra rules are auto-injected into the next generation |

**3 monkey-patches**: `build_user_prompt` (inject instructions) · `plot_piano_roll` (SSE broadcast) · `write_block_to_session` (slot tracking)

In [ ]:
import queue as _queue
from flask import Flask as _Flask, jsonify, Response, request, send_file
import logging as _flog
_flog.getLogger("werkzeug").setLevel(_flog.ERROR)

WEB_PORT = 5056

# ════════════════════════════════════════════════════════════════
# Shared state
# ════════════════════════════════════════════════════════════════
_web_config = {**CONFIG, "chord_progression": list(CONFIG["chord_progression"])}
_web_extra_rules: list = []
_web_status = {"running": False, "block": 0, "total": 0, "style": "", "slot": 0}
_web_figures: list = []
_web_sse_queues: list = []
_web_session_thread = None
_web_looper_ref = [None]   # mutable container, accessed from outside
_wlock = threading.Lock()

def _broadcast(etype: str, data: dict):
    msg = json.dumps({"type": etype, "data": data})
    dead = []
    for q in _web_sse_queues:
        try: q.put_nowait(msg)
        except: dead.append(q)
    for q in dead:
        try: _web_sse_queues.remove(q)
        except: pass

# ════════════════════════════════════════════════════════════════
# Monkey-patches (apply to every generation run after this cell)
# ════════════════════════════════════════════════════════════════
_orig_build_prompt = build_user_prompt

def build_user_prompt(config: dict, track_layout: list, style: str = "classic_groove") -> str:
    base = _orig_build_prompt(config, track_layout, style)
    with _wlock:
        rules = list(_web_extra_rules)
        _web_extra_rules.clear()
    if rules:
        inject = "\nUSER DIRECTOR INSTRUCTIONS (highest priority — override style rules if needed):\n"
        inject += "\n".join(f"- {r}" for r in rules) + "\n"
        base = base.replace("\nOutput ONLY this JSON", inject + "\nOutput ONLY this JSON")
    return base

_orig_plot = plot_piano_roll

def plot_piano_roll(midi_data, config, track_layout, save_path, track_colors=None):
    _orig_plot(midi_data, config, track_layout, save_path, track_colors)
    style_tag = midi_data.get("meta", {}).get("style", "")
    fig_info = {
        "filename":  save_path.name,
        "rel_path":  str(save_path.relative_to(OUTPUT_DIR)),
        "style":     style_tag,
        "notes":     sum(len(t["notes"]) for t in midi_data["tracks"]),
        "timestamp": datetime.datetime.now().isoformat(),
        "bpm":       config.get("bpm"),
        "key":       config.get("key"),
    }
    with _wlock:
        _web_figures.append(fig_info)
        _web_status["block"] = len(_web_figures)
        _web_status["style"] = style_tag
    _broadcast("figure_ready", fig_info)
    _broadcast("status", dict(_web_status))

_orig_write = write_block_to_session

def write_block_to_session(midi_data, slot_index, config, track_layout, **kw):
    _orig_write(midi_data, slot_index, config, track_layout, **kw)
    with _wlock:
        _web_status["slot"] = slot_index

# ════════════════════════════════════════════════════════════════
# Session runner (web UI -> background thread)
# ════════════════════════════════════════════════════════════════
def _web_run(cfg, n_iter, styles, start_slot, drift_pct):
    with _wlock:
        _web_status.update(running=True, block=0, total=(n_iter or 0), style="", slot=start_slot)
    _broadcast("status", dict(_web_status))
    try:
        run_live_session(
            cfg, TRACK_LAYOUT,
            n_iterations=n_iter,
            styles=styles,
            start_slot=start_slot,
            sm=synth_manager,
            drift_pct=drift_pct,
            lead_in=3.0,
        )
    finally:
        with _wlock:
            _web_status["running"] = False
        _broadcast("status", dict(_web_status))

# ════════════════════════════════════════════════════════════════
# Flask app
# ════════════════════════════════════════════════════════════════
_app = _Flask(__name__)
_app.logger.disabled = True

@_app.route("/")
def _index():
    return Response(_DASHBOARD_HTML, mimetype="text/html")

@_app.route("/figures/<path:filename>")
def _serve_fig(filename):
    p = OUTPUT_DIR / filename
    if not p.exists():
        return ("not found", 404)
    mime = "text/html" if p.suffix.lower() == ".html" else "image/png"
    return send_file(str(p), mimetype=mime)

@_app.route("/api/config", methods=["GET"])
def _get_cfg():
    with _wlock:
        out = dict(_web_config)
    out["_outputDir"] = str(OUTPUT_DIR.resolve())
    out["_styleKeys"] = list(PROMPT_STYLES.keys())
    out["_styleLabels"] = {k: v["label"] for k, v in PROMPT_STYLES.items()}
    return jsonify(out)

@_app.route("/api/config", methods=["POST"])
def _set_cfg():
    data = request.json or {}
    with _wlock:
        for field in ("bpm","key","genre","mood","bars"):
            if field in data:
                _web_config[field] = int(data[field]) if field in ("bpm","bars") else str(data[field])
        if "chord_progression" in data:
            _web_config["chord_progression"] = list(data["chord_progression"])[:4]
    if "bpm" in data and osc_client is not None:
        try: osc_client.send_message("/live/song/set/tempo", [float(_web_config["bpm"])])
        except Exception: pass
    _broadcast("config_updated", dict(_web_config))
    return jsonify({"ok": True})

@_app.route("/api/session/start", methods=["POST"])
def _start():
    global _web_session_thread
    with _wlock:
        if _web_status["running"]:
            return jsonify({"error": "already running"}), 409
        cfg = {**_web_config, "chord_progression": list(_web_config["chord_progression"]),
               "time_sig": tuple(CONFIG["time_sig"])}
    data = request.json or {}
    styles = data.get("styles") or None
    _n = int(data.get("n_iterations", 0))
    _web_session_thread = threading.Thread(
        target=_web_run,
        args=(cfg, (_n if _n > 0 else None), styles,
              int(data.get("start_slot",0)), float(data.get("drift_pct",0.15))),
        daemon=True, name="web_session"
    )
    _web_session_thread.start()
    return jsonify({"ok": True})

@_app.route("/api/session/stop", methods=["POST"])
def _stop():
    ref = _web_looper_ref[0]
    if ref: ref.stop_event.set()
    return jsonify({"ok": True})

@_app.route("/api/figures", methods=["GET"])
def _figures():
    with _wlock: return jsonify(list(_web_figures))

@_app.route("/api/status", methods=["GET"])
def _status():
    with _wlock: return jsonify(dict(_web_status))

@_app.route("/api/chat", methods=["POST"])
def _chat():
    data = request.json or {}
    msg  = str(data.get("message","")).strip()
    if not msg: return jsonify({"error": "empty"}), 400

    sys_prompt = (
        "You are a music production assistant controlling a 10-track MIDI generation pipeline in Ableton Live. "
        "The user gives instructions about how to modify the next 16-bar block.\n"
        "Respond with:\n"
        "1. A brief human-readable reply (1-2 sentences, in English).\n"
        "2. On a new line, a JSON block: {\"extra_rules\": [\"rule1\", \"rule2\", ...]}\n"
        "Rules must be concrete and technical. Produce 2-5 rules.\n"
        "Example rule: 'Add syncopated 16th-note bass hits on beats 2+ and 4+'\n"
        "Output the JSON on its own line after your text."
    )
    with _wlock: snap = {**_web_config}
    user_msg = (
        f"Session: {snap['bpm']} BPM, {snap['key']}, genre={snap['genre']}, "
        f"mood={snap['mood']}, chords={snap['chord_progression']}.\n\nInstruction: {msg}"
    )
    try:
        resp = client.messages.create(
            model="claude-haiku-4-5", max_tokens=600,
            system=sys_prompt,
            messages=[{"role":"user","content":user_msg}],
        )
        text = resp.content[0].text.strip()
    except Exception as e:
        return jsonify({"error": str(e)}), 500

    rules = []
    try:
        js = text.rfind("{"); je = text.rfind("}") + 1
        if js != -1 and je > js:
            rules = json.loads(text[js:je]).get("extra_rules", [])
            reply = text[:js].strip()
        else: reply = text
    except: reply = text

    with _wlock:
        _web_extra_rules.clear()
        _web_extra_rules.extend(rules)
    return jsonify({"reply": reply, "rules": rules})

@_app.route("/stream")
def _stream():
    def gen():
        q = _queue.Queue(maxsize=64)
        _web_sse_queues.append(q)
        with _wlock:
            init = {"status": dict(_web_status), "figures": list(_web_figures)}
        yield f"data: {json.dumps({'type':'init','data':init})}\n\n"
        try:
            while True:
                try: yield f"data: {q.get(timeout=25)}\n\n"
                except _queue.Empty: yield ": ping\n\n"
        except GeneratorExit:
            try: _web_sse_queues.remove(q)
            except: pass
    return Response(gen(), mimetype="text/event-stream",
                    headers={"Cache-Control":"no-cache","X-Accel-Buffering":"no"})

# ════════════════════════════════════════════════════════════════
# HTML -- monochrome (black & white) dark theme
# ════════════════════════════════════════════════════════════════
_DASHBOARD_HTML = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Ableton × Claude · Live MIDI</title>
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@300;400;500;700&family=Roboto+Mono:wght@400;500&display=swap" rel="stylesheet">
<link href="https://fonts.googleapis.com/icon?family=Material+Icons+Round" rel="stylesheet">
<style>
:root{
  --pri:#EDEDED;--on-pri:#161616;--pri-con:#333333;--on-pri-con:#F2F2F2;
  --sec:#BFBFBF;--ter:#BFBFBF;--err:#D6D6D6;
  --bg:#0F0F0F;--surf:#0F0F0F;--surf-var:#3A3A3A;
  --surf-c:#181818;--surf-ch:#222222;--surf-chh:#2C2C2C;
  --on-surf:#ECECEC;--on-surf-v:#B8B8B8;
  --out:#8A8A8A;--out-v:#3A3A3A;
  --grn:#FFFFFF;--amb:#CFCFCF;
  --rad:12px;
}
*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
html,body{height:100%;font-family:'Roboto',sans-serif;background:var(--bg);color:var(--on-surf);overflow:hidden}

/* ── Shell ── */
.shell{display:flex;flex-direction:column;height:100vh}

/* ── Top Bar ── */
.topbar{
  display:flex;align-items:center;gap:12px;
  padding:0 20px;height:64px;flex-shrink:0;
  background:var(--surf-c);
  border-bottom:1px solid var(--out-v);z-index:10;
}
.topbar .ico{font-size:28px;color:var(--pri)}
.topbar h1{font-size:20px;font-weight:500;flex:1}
.topbar h1 span{color:var(--pri)}
.chip-status{
  display:flex;align-items:center;gap:6px;
  padding:6px 16px;border-radius:100px;font-size:13px;font-weight:500;
  background:var(--surf-chh);color:var(--on-surf-v);transition:all .3s;
}
.chip-status.run{background:#2E2E2E;color:var(--grn)}
.dot{width:8px;height:8px;border-radius:50%;background:currentColor}
.chip-status.run .dot{animation:pulse 1.2s ease infinite}
@keyframes pulse{0%,100%{opacity:1}50%{opacity:.35}}
.out-dir{font-size:11px;color:var(--out);font-family:'Roboto Mono',monospace;
  padding:4px 10px;border-radius:4px;background:var(--surf-ch);
  max-width:220px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}

/* ── Grid ── */
.grid{display:grid;grid-template-columns:272px 1fr 316px;flex:1;overflow:hidden}
.panel{display:flex;flex-direction:column;overflow:hidden;border-right:1px solid var(--out-v)}
.panel:last-child{border-right:none}

/* ── Panel Header ── */
.ph{
  display:flex;align-items:center;gap:10px;
  padding:14px 18px 10px;border-bottom:1px solid var(--out-v);flex-shrink:0;
}
.ph .mi{font-size:20px;color:var(--pri)}
.ph h2{font-size:14px;font-weight:500}
.ph .spacer{flex:1}

/* ── Scrollable body ── */
.pb{flex:1;overflow-y:auto;padding:14px 16px}
.pb::-webkit-scrollbar{width:4px}
.pb::-webkit-scrollbar-thumb{background:var(--out-v);border-radius:2px}

/* ── Config Panel ── */
.cfg{background:var(--surf-c)}
.fl{display:block;font-size:11px;font-weight:500;color:var(--pri);
  letter-spacing:.6px;text-transform:uppercase;margin-bottom:6px}
.fg{margin-bottom:16px}
.mf{
  width:100%;background:var(--surf-ch);
  border:1px solid var(--out-v);border-radius:8px;
  color:var(--on-surf);font-family:'Roboto',sans-serif;font-size:14px;
  padding:9px 12px;outline:none;transition:border-color .2s;resize:none;
}
.mf:focus{border-color:var(--pri)}
select.mf{appearance:none;cursor:pointer}
textarea.mf{min-height:58px}
.chord-g{display:grid;grid-template-columns:1fr 1fr;gap:8px}
.chord-g .mf{text-align:center;font-family:'Roboto Mono',monospace;font-size:13px}
.sl-row{display:flex;align-items:center;gap:10px}
.sl{flex:1;-webkit-appearance:none;appearance:none;height:4px;border-radius:2px;
  background:var(--out-v);outline:none;cursor:pointer}
.sl::-webkit-slider-thumb{-webkit-appearance:none;width:20px;height:20px;border-radius:50%;
  background:var(--pri);cursor:pointer;box-shadow:0 2px 6px rgba(0,0,0,.4)}
.sl-v{font-size:12px;font-family:'Roboto Mono',monospace;width:36px;
  text-align:right;color:var(--on-surf-v)}
.num-g{display:grid;grid-template-columns:1fr 1fr;gap:8px}
.divider{height:1px;background:var(--out-v);margin:14px 0}

/* Style chips */
.scs{display:flex;flex-wrap:wrap;gap:6px;margin-top:2px}
.sc{padding:5px 11px;border-radius:100px;font-size:11px;cursor:pointer;
  border:1px solid var(--out-v);background:transparent;
  color:var(--on-surf-v);transition:all .15s;user-select:none}
.sc.on{background:var(--pri-con);color:var(--on-pri-con);border-color:var(--pri)}
.sc:hover{border-color:var(--pri)}

/* Buttons */
.btn{display:flex;align-items:center;justify-content:center;gap:6px;
  padding:11px 18px;border-radius:100px;border:none;cursor:pointer;
  font-family:'Roboto',sans-serif;font-size:14px;font-weight:500;
  letter-spacing:.1px;transition:all .2s;width:100%}
.btn-fill{background:var(--pri);color:var(--on-pri)}
.btn-fill:hover{box-shadow:0 2px 8px rgba(255,255,255,.18)}
.btn-fill:disabled{background:var(--out-v);color:var(--on-surf-v);cursor:not-allowed}
.btn-ton{background:var(--surf-chh);color:var(--on-surf-v)}
.btn-ton:hover{background:var(--out-v)}
.btn-out{background:transparent;color:var(--err);border:1px solid var(--err)}
.btns{display:flex;flex-direction:column;gap:8px;margin-top:4px}

/* ── Figure Panel ── */
.fig-panel{background:var(--surf)}

.slot-bar{
  display:flex;align-items:center;gap:8px;
  padding:8px 16px 0;flex-shrink:0;
}
.slot-scroll{display:flex;gap:6px;flex:1;overflow-x:auto;padding-bottom:8px}
.slot-scroll::-webkit-scrollbar{height:3px}
.slot-scroll::-webkit-scrollbar-thumb{background:var(--out-v);border-radius:2px}
.slot-c{flex-shrink:0;padding:4px 11px;border-radius:100px;font-size:11px;
  cursor:pointer;border:1px solid var(--out-v);background:transparent;
  color:var(--on-surf-v);transition:all .15s;white-space:nowrap}
.slot-c.act{background:var(--pri-con);color:var(--on-pri-con);border-color:var(--pri)}
.nav-btn{display:flex;align-items:center;justify-content:center;
  width:34px;height:34px;border-radius:50%;border:1px solid var(--out-v);
  background:transparent;color:var(--on-surf-v);cursor:pointer;flex-shrink:0;transition:all .15s}
.nav-btn:hover{background:var(--surf-ch);border-color:var(--pri);color:var(--pri)}

.fig-display{flex:1;overflow:auto;display:flex;align-items:center;
  justify-content:center;padding:12px;min-height:0}
.fig-display iframe{width:100%;height:100%;border:none;border-radius:8px;background:#0f0f0f;
  box-shadow:0 4px 24px rgba(0,0,0,.5)}
.placeholder{display:flex;flex-direction:column;align-items:center;gap:14px;
  color:var(--out)}
.placeholder .big{font-size:64px}
.placeholder p{font-size:15px}
.placeholder .sub{font-size:13px;color:var(--out-v)}

.meta-bar{
  display:flex;align-items:center;gap:12px;flex-wrap:wrap;
  padding:9px 16px;border-top:1px solid var(--out-v);
  flex-shrink:0;font-size:12px;color:var(--on-surf-v);
  font-family:'Roboto Mono',monospace;
}
.mt{display:flex;align-items:center;gap:5px;background:var(--surf-c);
  padding:3px 9px;border-radius:100px}
.mt .mi{font-size:13px}
.dl-btn{display:flex;align-items:center;gap:5px;padding:5px 12px;border-radius:100px;
  background:var(--surf-ch);border:none;color:var(--on-surf-v);font-size:11px;
  cursor:pointer;transition:all .15s;font-family:'Roboto',sans-serif}
.dl-btn:hover{background:var(--out-v)}

/* ── Chat Panel ── */
.chat-panel{background:var(--surf-c)}

.hints{display:flex;gap:6px;flex-wrap:wrap;
  padding:10px 16px 10px;border-bottom:1px solid var(--out-v);flex-shrink:0}
.hint{font-size:11px;padding:4px 10px;border-radius:100px;cursor:pointer;
  border:1px solid var(--out-v);color:var(--on-surf-v);background:transparent;
  transition:all .15s;white-space:nowrap}
.hint:hover{border-color:var(--pri);color:var(--pri)}

.msgs{flex:1;overflow-y:auto;padding:12px 14px;
  display:flex;flex-direction:column;gap:8px;min-height:0}
.msgs::-webkit-scrollbar{width:3px}
.msgs::-webkit-scrollbar-thumb{background:var(--out-v);border-radius:2px}

.msg{max-width:92%;padding:9px 13px;border-radius:12px;font-size:13px;
  line-height:1.55;animation:mi .18s ease}
@keyframes mi{from{opacity:0;transform:translateY(5px)}to{opacity:1;transform:none}}
.msg.user{align-self:flex-end;background:var(--pri-con);color:var(--on-pri-con);
  border-radius:12px 12px 3px 12px}
.msg.assistant{align-self:flex-start;background:var(--surf-chh);
  color:var(--on-surf);border-radius:3px 12px 12px 12px}
.msg.applied{align-self:center;font-size:11px;color:var(--grn);
  background:rgba(255,255,255,.08);padding:5px 12px;border-radius:100px}
.msg.sys{align-self:center;font-size:11px;color:var(--out);font-style:italic}
.rules-ul{margin-top:8px;list-style:none}
.rules-ul li{font-size:11px;font-family:'Roboto Mono',monospace;
  color:var(--pri);padding:2px 0}
.rules-ul li::before{content:"▸ "}

.cin{display:flex;flex-direction:column;gap:8px;
  padding:10px 14px;border-top:1px solid var(--out-v);flex-shrink:0}
.cta{width:100%;min-height:68px;max-height:110px;resize:none;
  background:var(--surf-ch);border:1px solid var(--out-v);border-radius:12px;
  color:var(--on-surf);font-family:'Roboto',sans-serif;font-size:13px;
  padding:9px 13px;outline:none;transition:border-color .2s}
.cta:focus{border-color:var(--pri)}
.cta::placeholder{color:var(--out)}
.ca{display:flex;gap:8px;justify-content:flex-end;align-items:center}
.send-btn{display:flex;align-items:center;justify-content:center;
  width:40px;height:40px;border-radius:50%;border:none;cursor:pointer;
  background:var(--pri);color:var(--on-pri);transition:all .15s}
.send-btn:hover{box-shadow:0 2px 8px rgba(255,255,255,.22)}
.send-btn:disabled{background:var(--out-v);color:var(--on-surf-v);cursor:not-allowed}
.clr-btn{display:flex;align-items:center;justify-content:center;
  width:34px;height:34px;border-radius:50%;border:1px solid var(--out-v);
  background:transparent;color:var(--on-surf-v);cursor:pointer;transition:all .15s}
.clr-btn:hover{border-color:var(--err);color:var(--err)}
.hint-line{font-size:11px;color:var(--out);text-align:center}

/* ── Snackbar ── */
.snack{position:fixed;bottom:24px;left:50%;transform:translateX(-50%) translateY(80px);
  background:#E6E1E5;color:#1C1B1F;padding:12px 20px;border-radius:8px;
  font-size:14px;z-index:9999;transition:transform .3s ease;pointer-events:none;
  box-shadow:0 4px 16px rgba(0,0,0,.5)}
.snack.show{transform:translateX(-50%) translateY(0)}

/* Progress bar */
.prog{height:3px;background:var(--pri-con);flex-shrink:0;
  transition:width .5s ease;border-radius:0 2px 2px 0}
</style>
</head>
<body>
<div class="shell">

<!-- Top Bar -->
<header class="topbar">
  <span class="material-icons-round ico">piano</span>
  <h1>Ableton &times; <span>Claude</span> &middot; Live MIDI</h1>
  <div class="chip-status" id="sc"><div class="dot"></div><span id="st">Idle</span></div>
  <div class="out-dir" id="outDir">...</div>
</header>
<div class="prog" id="prog" style="width:0%"></div>

<!-- Grid -->
<div class="grid" style="flex:1;overflow:hidden">

<!-- ① Config -->
<aside class="panel cfg">
  <div class="ph"><span class="material-icons-round mi">tune</span><h2>Configuration</h2></div>
  <div class="pb">
    <div class="fg">
      <label class="fl">BPM</label>
      <div class="sl-row">
        <input type="range" class="sl" id="bpmSl" min="60" max="200" value="124">
        <span class="sl-v" id="bpmV">124</span>
      </div>
    </div>
    <div class="fg">
      <label class="fl">Key</label>
      <select class="mf" id="keySel">
        <option>A minor</option><option>A major</option>
        <option>Bb minor</option><option>Bb major</option>
        <option>B minor</option><option>B major</option>
        <option>C minor</option><option>C major</option>
        <option>C# minor</option><option>D minor</option><option>D major</option>
        <option>Eb minor</option><option>Eb major</option>
        <option>E minor</option><option>E major</option>
        <option>F minor</option><option>F major</option>
        <option>F# minor</option><option>G minor</option><option>G major</option>
      </select>
    </div>
    <div class="fg">
      <label class="fl">Bars / Block</label>
      <select class="mf" id="barsSel">
        <option value="8">8 bars</option>
        <option value="16" selected>16 bars</option>
        <option value="32">32 bars</option>
      </select>
    </div>
    <div class="fg">
      <label class="fl">Genre</label>
      <textarea class="mf" id="genreF" rows="2"></textarea>
    </div>
    <div class="fg">
      <label class="fl">Mood</label>
      <textarea class="mf" id="moodF" rows="2"></textarea>
    </div>
    <div class="fg">
      <label class="fl">Chord Progression</label>
      <div class="chord-g">
        <input class="mf" id="c0" placeholder="bars 1-4">
        <input class="mf" id="c1" placeholder="bars 5-8">
        <input class="mf" id="c2" placeholder="bars 9-12">
        <input class="mf" id="c3" placeholder="bars 13-16">
      </div>
    </div>
    <button class="btn btn-ton" onclick="applyConfig()" style="margin-bottom:14px">
      <span class="material-icons-round" style="font-size:18px">check</span>Apply Config
    </button>
    <div class="divider"></div>
    <div class="fg">
      <label class="fl">Style Sequence</label>
      <div class="scs" id="scChips"></div>
    </div>
    <div class="fg">
      <label class="fl">Iterations &amp; Start Slot</label>
      <div class="num-g">
        <input type="number" class="mf" id="nIter" value="0" min="0" max="64" title="Iterations (0 = endless)">
        <input type="number" class="mf" id="sSlot" value="2" min="0" max="99" title="Start Slot">
      </div>
    </div>
    <div class="fg">
      <label class="fl">Drift &mdash; <span id="driftV">40</span>%</label>
      <input type="range" class="sl" id="driftSl" min="1" max="80" value="40">
    </div>
    <div class="btns">
      <button class="btn btn-fill" id="startBtn" onclick="startSess()">
        <span class="material-icons-round" style="font-size:18px">play_arrow</span>Start Session
      </button>
      <button class="btn btn-out" id="stopBtn" onclick="stopSess()" disabled>
        <span class="material-icons-round" style="font-size:18px">stop</span>Stop
      </button>
    </div>
  </div>
</aside>

<!-- (2) Piano Roll (wide) -->
<section class="panel fig-panel" style="display:flex;flex-direction:column">
  <div class="ph">
    <span class="material-icons-round mi">queue_music</span>
    <h2>Piano Roll</h2>
    <div class="spacer"></div>
    <span id="blkCnt" style="font-size:12px;color:var(--out);font-family:'Roboto Mono',monospace"></span>
  </div>
  <div class="slot-bar">
    <div class="slot-scroll" id="slotScroll"></div>
    <button class="nav-btn" onclick="nav(-1)" title="Prev"><span class="material-icons-round" style="font-size:18px">chevron_left</span></button>
    <button class="nav-btn" onclick="nav(1)"  title="Next"><span class="material-icons-round" style="font-size:18px">chevron_right</span></button>
  </div>
  <div class="fig-display" id="figDisp">
    <div class="placeholder">
      <span class="material-icons-round big">graphic_eq</span>
      <p>Start a session to generate MIDI</p>
      <p class="sub">Piano rolls appear here in real-time</p>
    </div>
  </div>
  <div class="meta-bar" id="metaBar" style="display:none">
    <div class="mt"><span class="material-icons-round mi">style</span><span id="mStyle">—</span></div>
    <div class="mt"><span class="material-icons-round mi">music_note</span><span id="mNotes">—</span>&nbsp;notes</div>
    <div class="mt"><span class="material-icons-round mi">speed</span><span id="mBpm">—</span>&nbsp;BPM</div>
    <div class="mt"><span class="material-icons-round mi">key</span><span id="mKey">—</span></div>
    <div class="mt"><span class="material-icons-round mi">schedule</span><span id="mTime">—</span></div>
    <div style="flex:1"></div>
    <button class="dl-btn" onclick="dlFig()">
      <span class="material-icons-round" style="font-size:14px">download</span>HTML
    </button>
  </div>
</section>

<!-- ③ Chat -->
<aside class="panel chat-panel" style="display:flex;flex-direction:column">
  <div class="ph">
    <span class="material-icons-round mi">smart_toy</span>
    <h2>MIDI Director</h2>
  </div>
  <div class="hints" id="hints"></div>
  <div class="msgs" id="msgs"></div>
  <div class="cin">
    <textarea class="cta" id="cta"
      placeholder="e.g. darker bass, add polyrhythm to the drums..."
      onkeydown="ckd(event)"></textarea>
    <div class="ca">
      <button class="clr-btn" onclick="clrChat()" title="Clear">
        <span class="material-icons-round" style="font-size:18px">clear</span>
      </button>
      <button class="send-btn" id="sendBtn" onclick="sendMsg()" title="Ctrl+Enter">
        <span class="material-icons-round" style="font-size:20px">send</span>
      </button>
    </div>
    <p class="hint-line">Ctrl+Enter &middot; rules apply to next generated block</p>
  </div>
</aside>

</div><!-- /grid -->
</div><!-- /shell -->
<div class="snack" id="snack"></div>

<script>
// ─── State ───────────────────────────────────────────────
let figs = [], curF = -1;
let selStyles = ["ambient_pad","sparse_minimal","classic_groove","build_up",
                  "bass_driven","breakbeat","glitch_micro","ambient_pad"];
let SKEYS = [], SLABELS = {};

const HINTS = [
  "Make the bass more syncopated and darker",
  "Add polyrhythm to the drums",
  "Make everything more minimal",
  "Make the lead more melodic",
  "Add glitch effects",
  "Gradually build the energy",
  "Make the chords more lush",
  "Make the percussion more organic",
];

// ─── Init ────────────────────────────────────────────────
window.addEventListener('DOMContentLoaded', async () => {
  addMsg('sys', 'Claude MIDI Director ready -- type an instruction for the next block.');
  buildHints();

  const cfg = await fetch('/api/config').then(r=>r.json()).catch(()=>null);
  if (cfg) {
    SKEYS = cfg._styleKeys || [];
    SLABELS = cfg._styleLabels || {};
    fillCfg(cfg);
    buildStyleChips();
  }

  const es = new EventSource('/stream');
  es.onmessage = e => onSSE(JSON.parse(e.data));
  es.onerror   = () => setStatus(false);

  document.getElementById('bpmSl').oninput   = e => document.getElementById('bpmV').textContent   = e.target.value;
  document.getElementById('driftSl').oninput = e => document.getElementById('driftV').textContent = e.target.value;
});

// ─── Config ──────────────────────────────────────────────
function fillCfg(c) {
  el('bpmSl').value   = c.bpm   || 124;
  el('bpmV').textContent = c.bpm || 124;
  selOpt('keySel',   c.key);
  selOpt('barsSel',  String(c.bars));
  el('genreF').value  = c.genre || '';
  el('moodF').value   = c.mood  || '';
  (c.chord_progression||[]).forEach((v,i)=>{ if(el('c'+i)) el('c'+i).value=v; });
  if (c._outputDir) { el('outDir').textContent=c._outputDir; el('outDir').title=c._outputDir; }
}

async function applyConfig() {
  const body = {
    bpm:  parseInt(el('bpmSl').value),
    key:  el('keySel').value,
    bars: parseInt(el('barsSel').value),
    genre: el('genreF').value,
    mood:  el('moodF').value,
    chord_progression: [0,1,2,3].map(i=>el('c'+i).value),
  };
  await fetch('/api/config',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify(body)});
  snack('Config applied ✓');
}

// ─── Style chips ─────────────────────────────────────────
function buildStyleChips() {
  const c = el('scChips'); c.innerHTML = '';
  SKEYS.forEach(k => {
    const b = document.createElement('button');
    b.className = 'sc' + (selStyles.includes(k) ? ' on' : '');
    b.textContent = SLABELS[k] || k;
    b.onclick = () => {
      const i = selStyles.indexOf(k);
      if (i===-1) { selStyles.push(k); b.classList.add('on'); }
      else { selStyles.splice(i,1); b.classList.remove('on'); }
    };
    c.appendChild(b);
  });
}

// ─── Hints ───────────────────────────────────────────────
function buildHints() {
  const c = el('hints');
  HINTS.forEach(h => {
    const b = document.createElement('button');
    b.className = 'hint'; b.textContent = h;
    b.onclick = () => el('cta').value = h;
    c.appendChild(b);
  });
}

// ─── Session ─────────────────────────────────────────────
async function startSess() {
  const body = {
    n_iterations: parseInt(el('nIter').value),
    start_slot:   parseInt(el('sSlot').value),
    drift_pct:    parseInt(el('driftSl').value)/100,
    styles:       selStyles.length ? selStyles : null,
  };
  const r = await fetch('/api/session/start',{method:'POST',
    headers:{'Content-Type':'application/json'},body:JSON.stringify(body)});
  if (r.ok) snack('Session starting… press Play in Ableton ▶');
  else { const e=await r.json(); snack('Error: '+(e.error||'?')); }
}

async function stopSess() {
  await fetch('/api/session/stop',{method:'POST'});
  snack('Stop signal sent');
}

// ─── SSE ─────────────────────────────────────────────────
function onSSE(msg) {
  const {type, data} = msg;
  if (type==='init') {
    if (data.status) updStatus(data.status);
    if (data.figures && data.figures.length) {
      figs = data.figures;
      rebuildSlots();
      showFig(figs.length-1);
    }
  } else if (type==='figure_ready') {
    figs.push(data);
    rebuildSlots();
    showFig(figs.length-1);
    snack('New block ready: ' + data.style);
  } else if (type==='status') {
    updStatus(data);
  } else if (type==='config_updated') {
    fillCfg(data);
  }
}

function updStatus(s) {
  const chip = el('sc'), txt = el('st');
  if (s.running) {
    chip.classList.add('run');
    txt.textContent = `Block ${s.block}/${s.total}` + (s.style ? ' · '+s.style : '');
  } else {
    chip.classList.remove('run');
    txt.textContent = 'Idle';
  }
  el('startBtn').disabled = !!s.running;
  el('stopBtn').disabled  = !s.running;
  if (s.total > 0) {
    el('blkCnt').textContent = s.block + ' / ' + s.total;
    el('prog').style.width = (s.block/s.total*100) + '%';
  }
}

function setStatus(on) {
  if (!on) { el('sc').classList.remove('run'); el('st').textContent='Disconnected'; }
}

// ─── Figures ─────────────────────────────────────────────
function rebuildSlots() {
  const c = el('slotScroll'); c.innerHTML = '';
  figs.forEach((f,i) => {
    const b = document.createElement('button');
    b.className = 'slot-c' + (i===curF ? ' act' : '');
    b.textContent = (i+1) + ' · ' + f.style;
    b.onclick = () => showFig(i);
    c.appendChild(b);
  });
}

function showFig(idx) {
  if (idx<0||idx>=figs.length) return;
  curF = idx;
  const f = figs[idx];
  el('figDisp').innerHTML =
    `<iframe src="/figures/${f.rel_path}?t=${Date.now()}" title="piano roll"></iframe>`;
  el('metaBar').style.display = 'flex';
  el('mStyle').textContent = f.style;
  el('mNotes').textContent = f.notes;
  el('mBpm').textContent   = f.bpm || '—';
  el('mKey').textContent   = f.key || '—';
  el('mTime').textContent  = (f.timestamp||'').slice(11,19);
  document.querySelectorAll('.slot-c').forEach((b,i)=>b.classList.toggle('act',i===idx));
  // scroll into view
  const s = el('slotScroll');
  const bs = s.querySelectorAll('.slot-c');
  if (bs[idx]) bs[idx].scrollIntoView({behavior:'smooth',inline:'nearest'});
}

function nav(d) { showFig(curF+d); }

function dlFig() {
  if (curF<0) return;
  const f = figs[curF];
  const a = document.createElement('a');
  a.href = `/figures/${f.rel_path}`;
  a.download = f.filename;
  a.click();
}

// ─── Chat ────────────────────────────────────────────────
function ckd(e) { if (e.ctrlKey && e.key==='Enter') { e.preventDefault(); sendMsg(); } }

async function sendMsg() {
  const inp = el('cta');
  const msg = inp.value.trim();
  if (!msg) return;
  addMsg('user', esc(msg));
  inp.value = '';
  el('sendBtn').disabled = true;
  addMsg('assistant', '<span style="color:var(--out)">Thinking…</span>', 'thinking');

  try {
    const r = await fetch('/api/chat',{method:'POST',
      headers:{'Content-Type':'application/json'},body:JSON.stringify({message:msg})});
    const d = await r.json();
    el('thinking')?.remove();
    if (d.error) {
      addMsg('assistant','⚠️ '+esc(d.error));
    } else {
      let html = `<div>${esc(d.reply)}</div>`;
      if (d.rules?.length) {
        html += '<ul class="rules-ul">' + d.rules.map(r=>`<li>${esc(r)}</li>`).join('') + '</ul>';
        addMsg('assistant', html);
        addMsg('applied', `✓ ${d.rules.length} rule${d.rules.length>1?'s':''} queued for next block`);
        snack(`${d.rules.length} rules queued ✓`);
      } else { addMsg('assistant', html); }
    }
  } catch(e) {
    el('thinking')?.remove();
    addMsg('assistant','⚠️ Network error');
  }
  el('sendBtn').disabled = false;
}

function addMsg(cls, html, id) {
  const c = el('msgs');
  const d = document.createElement('div');
  d.className = 'msg '+cls;
  if (id) d.id = id;
  d.innerHTML = html;
  c.appendChild(d);
  c.scrollTop = c.scrollHeight;
}

function clrChat() { el('msgs').innerHTML=''; addMsg('sys','Chat cleared.'); }

// ─── Utils ───────────────────────────────────────────────
const el = id => document.getElementById(id);
function selOpt(selId, val) {
  [...el(selId).options].forEach(o => { if(o.value===val||o.text===val) o.selected=true; });
}
function esc(s) {
  return (s||'').replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;').replace(/\n/g,'<br>');
}
let _snackT;
function snack(msg, ms=3000) {
  const e=el('snack'); e.textContent=msg; e.classList.add('show');
  clearTimeout(_snackT); _snackT=setTimeout(()=>e.classList.remove('show'),ms);
}
</script>
</body>
</html>"""

# ════════════════════════════════════════════════════════════════
# Start server
# ════════════════════════════════════════════════════════════════
if not any(t.name == "web_dash" for t in threading.enumerate()):
    threading.Thread(
        target=lambda: _app.run(port=WEB_PORT, threaded=True, use_reloader=False),
        name="web_dash", daemon=True,
    ).start()
    print(f"OK  web dashboard started: http://127.0.0.1:{WEB_PORT}")
    print(f"   Output: {OUTPUT_DIR.resolve()}")
else:
    print(f"OK  already running: http://127.0.0.1:{WEB_PORT}")
print("OK  monkey-patches applied (build_user_prompt / plot_piano_roll / write_block_to_session)")

OK  web dashboard started: http://127.0.0.1:5056
   Output: /Users/faacia/longlive/outputs/20260601_172101
OK  monkey-patches applied (build_user_prompt / plot_piano_roll / write_block_to_session)
 * Serving Flask app '__main__'
 * Debug mode: off


Block: 16 bars = 28.7s @ 134 BPM
  OSC          : OFF
  SynthManager : OFF
  Styles       : ['ambient_pad', 'sparse_minimal', 'classic_groove', 'build_up', 'bass_driven', 'breakbeat', 'glitch_micro', 'ambient_pad']
  Camelot       : ON (start 8A, +1/block)
  Iterations   : ENDLESS
  Buffer ahead : 2 block(s)
[gen] style=ambient_pad (Ambient / atmospheric) model=claude-opus-4-7
      streaming…
[Pre-roll] generating first block(s)...
> block 1  slot=3  key=A minor  8A  style=ambient_pad  notes=66
 70.4s, 15365 chars, stop=end_turn
[gen] style=classic_groove (Classic locked groove) model=claude-opus-4-7
      streaming…> block 2  slot=4  key=E minor  9A  style=sparse_minimal  notes=211
[gen] style=classic_groove (Classic locked groove) model=claude-opus-4-7
      streaming… 185.1s, 41770 chars, stop=end_turn
✓ Session column 3: '03 · Am9 · classic_groove'
[gen] style=bass_driven (Bass-driven groove) model=claude-opus-4-7
      streaming… 222.0s, 50021 chars, stop=end_turn
✓ Session colum

---
## Cell 12 · Run

Use the **Start Session** button on the web dashboard (http://127.0.0.1:5056),
or run the cell below directly.

In [17]:
# Endless, seamless DJ set: blocks are buffered ahead so playback never stops,
# the key climbs the Camelot wheel (+1 / block), and the synth timbre drifts hard.
# Press the Stop button (web) or interrupt the kernel to end.
run_live_session(
    CONFIG, TRACK_LAYOUT,
    n_iterations=None,          # None = endless
    styles=[
        "classic_groove", "bass_driven", "build_up", "polyrhythmic",
        "classic_groove", "breakbeat",   "bass_driven", "build_up",
    ],
    start_slot=2,
    sm=synth_manager,
    drift_pct=0.45,             # pronounced synth evolution
    buffer_ahead=2,
    rotate_camelot=True,
    lead_in=3.0,
)

# ─── other options ───────────────────────────────────────────
# # B. Fixed length (8 blocks)
# run_live_session(CONFIG, TRACK_LAYOUT, n_iterations=8,
#                  styles=None, sm=synth_manager, drift_pct=0.45)
#
# # C. Lock to one style, no key rotation
# run_live_session(CONFIG, TRACK_LAYOUT, n_iterations=None,
#                  styles="bass_driven", rotate_camelot=False, sm=synth_manager, drift_pct=0.5)
#
# # D. Even stronger synth motion
# run_live_session(CONFIG, TRACK_LAYOUT, n_iterations=None,
#                  sm=synth_manager, drift_pct=0.6, buffer_ahead=3)


Block: 16 bars = 28.7s @ 134 BPM
  OSC          : OFF
  SynthManager : OFF
  Styles       : ['classic_groove', 'bass_driven', 'build_up', 'polyrhythmic', 'classic_groove', 'breakbeat', 'bass_driven', 'build_up']
  Camelot       : ON (start 8A, +1/block)
  Iterations   : ENDLESS
  Buffer ahead : 2 block(s)

[Pre-roll] generating first block(s)...

> starting in 3s -- press Play in Ableton.

  -> clip fire: slot 3
> block 1  slot=3  key=A minor  8A  style=classic_groove  notes=591
  -> clip fire: slot 4
> block 2  slot=4  key=E minor  9A  style=bass_driven  notes=710
  -> clip fire: slot 5
> block 3  slot=5  key=B minor  10A  style=build_up  notes=501
  -> clip fire: slot 6
> block 4  slot=6  key=F# minor  11A  style=polyrhythmic  notes=517
  -> clip fire: slot 7
> block 5  slot=7  key=C# minor  12A  style=classic_groove  notes=712

[stopped]

OK done. Outputs: /Users/faacia/longlive/outputs/20260601_172101


## Cell 13 · Cleanup (emergency)

In [16]:
# === EMERGENCY HOTFIX: re-enable clip writing in the live kernel ===
def _orig_write(midi_data, slot_index, config, track_layout, overwrite=True, verbose=False):
    if osc_client is None:                      # <-- no longer requires lq
        if verbose: print("! OSC disabled"); 
        return
    bars = config["bars"]; length_beats = float(bars * config["time_sig"][0])
    first_chord = config["chord_progression"][0]
    style_tag = midi_data.get("meta", {}).get("style", "")
    clip_name = f"{slot_index+1:02d} · {first_chord}" + (f" · {style_tag}" if style_tag else "")
    clip_name = clip_name[:40]
    for track in midi_data["tracks"]:
        ti = track["track_index"]
        if overwrite:
            try: osc_client.send_message("/live/clip_slot/delete_clip", [ti, slot_index]); time.sleep(0.02)
            except Exception: pass
        osc_client.send_message("/live/clip_slot/create_clip", [ti, slot_index, length_beats]); time.sleep(0.04)
        osc_client.send_message("/live/clip/set/name", [ti, slot_index, clip_name]); time.sleep(0.02)
        notes_in_range = [n for n in track["notes"] if 0 <= n["start_beat"] < length_beats]
        if notes_in_range:
            lo, hi = track_layout[ti]["pitch_range"]; args = [ti, slot_index]
            for n in notes_in_range:
                args.extend([int(max(lo, min(hi, n["pitch"]))), float(n["start_beat"]),
                             max(0.0625, float(n["duration"])), int(max(1, min(127, n["velocity"]))), False])
            osc_client.send_message("/live/clip/add/notes", args); time.sleep(0.03)
    print(f"✓ Session column {slot_index+1}: '{clip_name}'")
print("✓ hotfix applied — clips will be written on the next block")


✓ hotfix applied — clips will be written on the next block


In [ ]:
# Force-release stuck notes
if midi_out is not None:
    for ch in range(16):
        midi_out.send(mido.Message("control_change", channel=ch, control=123, value=0))
    print("All notes off sent on channels 1-16.")

# On full session shutdown (commented out for safe re-runs -- call manually if needed)
# midi_out.close()
# _osc_server.shutdown(); _osc_server.server_close()